## **06 - Cleaning Data**
***

**What it does:** cleans each of the three tables on its own, before they are joined in `07_joined_table.ipynb`. No rows are filtered here; all row filters (US only, 3+ listings, price cap, non-concerts) are applied in 07, after the join.

**Input:** the `shows`, `artists` and `venues` tabs of the shared Google Sheet (outputs of notebooks 03-05)
**Output:** `../data/shows_clean.csv`, `../data/artists_clean.csv`, `../data/venues_clean.csv`

**Steps**
1. **Load** the three tables from the Google Sheet.
2. **Low-confidence Last.fm matches:** listed for review and **kept** (`FIX_BAD_MATCHES = False`). Most are correct matches under a different name (aliases, names in another script, co-headlined shows).
3. **Unmatched venues:** lists the busiest venues SeatGeek couldn't find.
4. **Fill missing venue locations (3b):**
   * First from SeatGeek-matched venues in the same city, when they all agree on country and state
   * Then from a hand-checked dictionary (`VENUE_LOCATIONS`, 284 venues). It also covers city names shared by several places (Portland OR/ME, Columbia MO/SC/MS, Birmingham AL/England), where the same-city match could be wrong
   * `location_source` records where each venue's location came from
5. **Fix time zones (3c):**
   * Cleans SeatGeek's time zones against each venue's state
   * Converts every show to its venue's local time, then rebuilds `weekday`, `is_weekend`, `month` and `show_hour`
6. **Duplicates (3d):**
   * Merges venue keys that SeatGeek matched to the same venue
   * Merges artist keys that Last.fm matched to the same artist (low-confidence matches excluded)
   * Drops repeated shows (same `event_id`, or same artist, venue and start time); early and late shows on the same night are kept
7. **Save** the three cleaned tables for 07.

**Not used:** venue capacity. It is known for only about 19% of shows, so it can't be a feature. 07 builds a venue-size stand-in (`venue_listings`) from the number of listings on the snapshot day instead.

**basic cleaning was done when data was pulled and tabled.** this includes things like festival passes and theme nights dropped, artist names pulled from show names, one price per show (about 14 days out), and duplicates removed. 

THE FOUR TABLES (TO BE JOINED)

1. data/music.db      ← raw Gametime (1.9M daily rows), only used to build shows
2. data/shows.csv     ← one row per show (29,613)
3. data/artists.csv   ← one row per artist (5,139)
4. data/venues.csv    ← one row per venue (after 05 finishes)

**cleaning agenda**

- Hand-fill venue capacity (split the work): data/manual/venue_capacity_manual.csv, about 150 venues. Each person takes about 40, looking up capacity on the venue’s website or Wikipedia, then rerun 05.
- Check bad artist matches: 31 low-confidence Last.fm matches listed at the end of 04 (e.g., “World of Warcraft”, “Lord of the Rings”). Fix or drop them.
- Check unmatched venues: the busiest venues SeatGeek couldn’t find, listed at the end of 05. Look for typos or wrong cities.
- Remove non-US shows: about 1,000 shows in London, Montreal and similar; filter using SeatGeek’s country column.
- Extreme prices: shows priced above about $2,000 (the top is $49,907). Look at them and decide whether they’re real or junk.
- Group decisions:
    - target: lowest price or median price
    - minimum listings: 40% of shows have fewer than 3
    - any other show types to drop (e.g., Las Vegas magic and comedy shows like Criss Angel and Tape Face that aren’t concerts)

In [25]:
# Setup: loading in data from shared Google Sheet
import numpy as np
import pandas as pd

SHEET_ID = "17uOoK1UZboRWShSb2OZz3GLdITGPXoXBN1k5IKE29A0"
SHEET_URL = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet="

shows = pd.read_csv(SHEET_URL + "shows", low_memory=False)
print(f"shows data has {len(shows)} observations")
artists = pd.read_csv(SHEET_URL + "artists", low_memory=False)
print(f"artists data has {len(artists)} observations")
venues = pd.read_csv(SHEET_URL + "venues", low_memory=False)
print(f"venues data has {len(venues)} observations")

shows data has 29613 observations
artists data has 5139 observations
venues data has 2722 observations


In [26]:
# 2. Bad artist matches (Last.fm matched the wrong artist)
# not dropping them yet because later we can fix them
FIX_BAD_MATCHES = False   # True = blank out Last.fm data for low-confidence matches

bad = artists["low_confidence_match"].fillna(False).astype(bool)
print(f"{bad.sum()} low-confidence Last.fm matches:")
display(artists.loc[bad, ["artist_name", "lastfm_name", "name_similarity", "listeners", "n_shows"]]
               .sort_values("n_shows", ascending=False))

if FIX_BAD_MATCHES:
    lastfm_cols = ["lastfm_name", "listeners", "log_listeners", "playcount",
                   "plays_per_listener", "on_tour", "tags", "top_tag"]
    artists[lastfm_cols] = artists[lastfm_cols].astype(object)   # lets the bool on_tour column hold NaN
    artists.loc[bad, lastfm_cols] = np.nan       # keep the artist, drop the wrong popularity data
    artists.loc[bad, "lastfm_status"] = "bad_match"
    print(f"Blanked Last.fm data for {bad.sum()} artists")

31 low-confidence Last.fm matches:


,artist_name,lastfm_name,name_similarity,listeners,n_shows
29,Musiq Soulchild,Musiq,0.500000,402237.0,51
642,The Lord of the Rings,Howard Shore,0.413793,1381300.0,20
895,Avenged Sevenfold and Good Charlotte,Good Charlotte,0.560000,2765373.0,16
1055,Eivor,Eivør Pálsdóttir,0.500000,30328.0,14
1240,Lady A,Lady Antebellum,0.571429,1118303.0,12
1461,Pat Benatar and Neil Giraldo,Pat Benatar,0.564103,1496558.0,10
1541,Sound Tribe Sector 9,STS9,0.333333,155869.0,10
1711,Joe Hisaishi,久石譲,0.000000,173483.0,9
2062,Morris Day and the Time,The Time,0.296296,170530.0,7
2152,Acid Mothers Temple,Acid Mothers Temple & The Melting Paraiso U.F.O.,0.558824,68926.0,6


In [27]:
# 3. Unmatched venues SeatGeek couldn't find
venues["sg_matched"] = venues["sg_venue_id"].notna()
unmatched = venues[~venues["sg_matched"]].sort_values("n_shows", ascending=False)
share = shows["venue_key"].isin(unmatched["venue_key"]).mean()
print(f"{len(unmatched):,} venues not matched ({np.round(share,3)*100}% of shows)")
print("Busiest unmatched venues (worth a quick look for typos or wrong cities):")
display(unmatched[["venue", "city", "n_shows", "median_max_listings"]].head(10))
# median_max_listings is still in as a rough replacement of venue size.

591 venues not matched (24.7% of shows)
Busiest unmatched venues (worth a quick look for typos or wrong cities):


,venue,city,n_shows,median_max_listings
2,House of Blues,Houston,93,1.0
6,Birdland,New York,81,1.0
14,The Hi Fi,Indianapolis,72,1.0
18,Eagles Ballroom,Milwaukee,67,1.0
24,History,Toronto,57,2.0
26,Revolution Hall,Portland,56,9.0
29,The Intersection,Grand Rapids,55,1.0
31,The Sinclair,Cambridge,55,1.0
40,Greek Theatre,Los Angeles,51,30.0
44,First Avenue,Minneapolis,51,1.0


### 3b. Fill in missing venue locations
About a quarter of venues have no country, state or time zone because SeatGeek could not match them. They are filled in three steps:
1. **SeatGeek, same city:** a missing venue takes the country, state and time zone of the SeatGeek-matched venues in its city, but only when all of those venues agree on country and state. The city comes from `venue_key`. SeatGeek's time zones are sometimes inconsistent within a city, so the most common one is used.
2. **Print** the venues that are still missing.
3. **Hand-checked dictionary (`VENUE_LOCATIONS`):** each remaining venue's location, looked up by hand (with AI). The dictionary covers venues in city names shared by several places (Portland OR/ME, Columbia MO/SC/MS,etc), where the city match in step 1 could be wrong. Where both overlap, the dictionary presides.

`location_source` records where each venue's location came from: `seatgeek`, `seatgeek_city`, `manual` or `missing`.

In [28]:
# 3b-1. Fill missing locations from SeatGeek venues in the same city
venues["city_key"] = venues["venue_key"].str.split("|").str[1]
venues["state"] = venues["state"].str.strip().str.upper()
venues["location_source"] = np.where(venues["country"].notna(), "seatgeek", "missing")

known = venues[venues["location_source"].eq("seatgeek")]
by_city = known.groupby("city_key").agg(
    n_country=("country", "nunique"), n_state=("state", "nunique"),
    country=("country", "first"), state=("state", "first"),
    timezone=("timezone", lambda s: s.mode().iat[0] if s.notna().any() else np.nan))
# use a city only when all its SeatGeek venues agree on country and state
agree = by_city[(by_city["n_country"] == 1) & (by_city["n_state"] <= 1)]

fill = venues["location_source"].eq("missing") & venues["city_key"].isin(agree.index)
for c in ["country", "state", "timezone"]:
    venues.loc[fill, c] = venues.loc[fill, "city_key"].map(agree[c])
venues.loc[fill, "location_source"] = "seatgeek_city"
print(f"Filled from SeatGeek venues in the same city: {fill.sum()} venues ({venues.loc[fill, 'n_shows'].sum():,} shows)")

Filled from SeatGeek venues in the same city: 372 venues (5,419 shows)


In [29]:
# 3b-2. Venues still missing a location (name|city), busiest first
still = venues[venues["location_source"].eq("missing")].sort_values("n_shows", ascending=False)
print(f"{len(still)} venues still missing ({still['n_shows'].sum():,} shows):")


222 venues still missing (1,968 shows):


**Hand-checked venue locations.** Each entry is `venue_key: (state, country, timezone)`. State is blank (`None`) outside the US. Country names follow SeatGeek's (`US`, `Canada`, `UK`, etc.). US time zones use the main zone for the area (for example `America/New_York` for Detroit so needs to be adjusted later). Venues marked with a comment were looked up because the city name alone was not enough.

In [30]:
# 3b-3. Hand-checked venue locations
VENUE_LOCATIONS = {
    # venue_key (name|city): (state, country, timezone)

    # --- outside the US (dropped by the US filter) ---
    "afas live|amsterdam": (None, "Netherlands", "Europe/Amsterdam"),
    "lotto arena|antwerp": (None, "Belgium", "Europe/Brussels"),
    "unipol forum|assago": (None, "Italy", "Europe/Rome"),
    "o2 belfast|belfast": (None, "UK", "Europe/London"),
    "velodrom berlin|berlin": (None, "Germany", "Europe/Berlin"),
    "bp pulse live at resorts world arena|birmingham": (None, "UK", "Europe/London"),  # Birmingham, England
    "utilita arena|birmingham": (None, "UK", "Europe/London"),  # Birmingham, England
    "estadio nemesio camacho|bogota": (None, "Colombia", "America/Bogota"),
    "tipos arena|bratislava": (None, "Slovakia", "Europe/Bratislava"),
    "ovb arena|bremen": (None, "Germany", "Europe/Berlin"),
    "le club square dix30|brossard": (None, "Canada", "America/Toronto"),
    "mvm dome|budapest": (None, "Hungary", "Europe/Budapest"),
    "commonwealth bar|calgary": (None, "Canada", "America/Edmonton"),
    "macewan hall|calgary": (None, "Canada", "America/Edmonton"),
    "palace theatre|calgary": (None, "Canada", "America/Edmonton"),
    "psd bank dome|dusseldorf": (None, "Germany", "Europe/Berlin"),
    "o2 academy edinburgh|edinburgh": (None, "UK", "Europe/London"),
    "scottish gas murrayfield stadium|edinburgh": (None, "UK", "Europe/London"),
    "fan park at ice district|edmonton": (None, "Canada", "America/Edmonton"),
    "starlite room|edmonton": (None, "Canada", "America/Edmonton"),
    "union hall|edmonton": (None, "Canada", "America/Edmonton"),
    "winspear centre|edmonton": (None, "Canada", "America/Edmonton"),
    "river cree casino|enoch": (None, "Canada", "America/Edmonton"),
    "festhalle frankfurt|frankfurt am main": (None, "Germany", "Europe/Berlin"),
    "lanxess arena|germany": (None, "Germany", "Europe/Berlin"),
    "ovo hydro|glasgow": (None, "UK", "Europe/London"),
    "hard rock hotel and casino ottawa|gloucester": (None, "Canada", "America/Toronto"),
    "barclays arena|hamburg": (None, "Germany", "Europe/Berlin"),
    "bridgeworks|hamilton": (None, "Canada", "America/Toronto"),
    "estadio do restelo|lisbon": (None, "Portugal", "Europe/Lisbon"),
    "m and s bank arena liverpool|liverpool": (None, "UK", "Europe/London"),
    "roundhouse|london": (None, "UK", "Europe/London"),
    "estadio shakira|madrid": (None, "Spain", "Europe/Madrid"),
    "movistar arena|madrid": (None, "Spain", "Europe/Madrid"),
    "ao arena|manchester": (None, "UK", "Europe/London"),
    "aviva studios|manchester": (None, "UK", "Europe/London"),
    "estadio gnp seguros|mexico city": (None, "Mexico", "America/Mexico_City"),
    "unipol dome|milano mi": (None, "Italy", "Europe/Rome"),
    "estadio bbva|monterrey": (None, "Mexico", "America/Monterrey"),
    "bell centre|montreal": (None, "Canada", "America/Toronto"),
    "club soda|montreal": (None, "Canada", "America/Toronto"),
    "l astral|montreal": (None, "Canada", "America/Toronto"),
    "metropolis|montreal": (None, "Canada", "America/Toronto"),
    "salle wilfrid pelletier place des arts|montreal": (None, "Canada", "America/Toronto"),
    "utilita arena newcastle upon tyne|newcastle upon tyne": (None, "UK", "Europe/London"),
    "sobeys stadium|north york": (None, "Canada", "America/Toronto"),
    "unity arena|oslo": (None, "Norway", "Europe/Oslo"),
    "southam hall at national art centre|ottawa": (None, "Canada", "America/Toronto"),
    "accor arena|paris": (None, "France", "Europe/Paris"),
    "o2 arena prague|prague": (None, "Czech Republic", "Europe/Prague"),
    "theatre capitole|quebec": (None, "Canada", "America/Toronto"),
    "stade de france|saint denis": (None, "France", "Europe/Paris"),
    "riyadh air metropolitano|san blas canillejas": (None, "Spain", "Europe/Madrid"),
    "coca cola music hall of puerto rico|san juan": (None, "Puerto Rico", "America/Puerto_Rico"),
    "estadio nacional julio martinez pradanos|santiago": (None, "Chile", "America/Santiago"),
    "utilita arena sheffield|sheffield": (None, "UK", "Europe/London"),
    "strawberry arena|solna": (None, "Sweden", "Europe/Stockholm"),
    "avicii arena|stockholm": (None, "Sweden", "Europe/Stockholm"),
    "drake hotel|toronto": (None, "Canada", "America/Toronto"),
    "history|toronto": (None, "Canada", "America/Toronto"),
    "horseshoe tavern|toronto": (None, "Canada", "America/Toronto"),
    "opera house|toronto": (None, "Canada", "America/Toronto"),
    "rogers stadium|toronto": (None, "Canada", "America/Toronto"),
    "estadio azteca|ursula coapa": (None, "Mexico", "America/Mexico_City"),
    "roig arena|valencia": (None, "Spain", "Europe/Madrid"),
    "biltmore cabaret|vancouver": (None, "Canada", "America/Vancouver"),
    "hollywood theatre|vancouver": (None, "Canada", "America/Vancouver"),
    "orpheum theatre|vancouver": (None, "Canada", "America/Vancouver"),
    "pearl|vancouver": (None, "Canada", "America/Vancouver"),
    "vogue theatre|vancouver": (None, "Canada", "America/Vancouver"),
    "royal theatre|victoria": (None, "Canada", "America/Vancouver"),
    "pge narodowy|warsaw": (None, "Poland", "Europe/Warsaw"),
    "burning kiln winery|williams": (None, "Canada", "America/Toronto"),  # St. Williams, ON
    "university of wolverhampton at the civic hall|wolverhampton": (None, "UK", "Europe/London"),

    # --- US: no SeatGeek venue in this city, so the city check could not fill these ---
    "abilene auditorium|abilene": ('TX', "US", "America/Chicago"),
    "moody coliseum|abilene": ('TX', "US", "America/Chicago"),
    "stephen talkhouse|amagansett": ('NY', "US", "America/New_York"),
    "american canyon ruins|american canyon": ('CA', "US", "America/Los_Angeles"),
    "arvada center outdoors amp|arvada": ('CO', "US", "America/Denver"),
    "paramount arts center|ashland": ('KY', "US", "America/New_York"),
    "belly up|aspen": ('CO', "US", "America/Denver"),
    "criterion theatre|bar harbor": ('ME', "US", "America/New_York"),
    "momentary|bentonville": ('AR', "US", "America/Chicago"),
    "stone church vt|brattleboro": ('VT', "US", "America/New_York"),
    "barnhill center hasskarl auditorium|brenham": ('TX', "US", "America/Chicago"),
    "cache creek event center|brooks": ('CA', "US", "America/Los_Angeles"),
    "tilles center concert hall|brookville": ('NY', "US", "America/New_York"),
    "sinclair|cambridge": ('MA', "US", "America/New_York"),
    "sonia|cambridge": ('MA', "US", "America/New_York"),
    "sunset center carmel by the sea|carmel": ('CA', "US", "America/Los_Angeles"),
    "cat s cradle|carrboro": ('NC', "US", "America/New_York"),
    "cerritos center for the performing arts|cerritos": ('CA', "US", "America/Los_Angeles"),
    "event center at hollywood casino|charles town": ('WV', "US", "America/New_York"),
    "harrah s cherokee|cherokee": ('NC', "US", "America/New_York"),
    "pine knob|clarkston": ('MI', "US", "America/New_York"),
    "surf ballroom and museum|clear lake": ('IA', "US", "America/Chicago"),
    "coachella crossroads spotlight 29 casino|coachella": ('CA', "US", "America/Los_Angeles"),
    "cow palace|daly city": ('CA', "US", "America/Los_Angeles"),
    "dania improv comedy theatre|dania beach": ('FL', "US", "America/New_York"),
    "mondavi center|davis": ('CA', "US", "America/Los_Angeles"),
    "club la destin|destin": ('FL', "US", "America/Chicago"),
    "grace church eden prairie|eden prairie": ('MN', "US", "America/Chicago"),
    "magnolia el cajon|el cajon": ('CA', "US", "America/Los_Angeles"),
    "old saloon|emigrant": ('MT', "US", "America/Denver"),
    "devils backbone tavern|fischer": ('TX', "US", "America/Chicago"),
    "majestic|fort smith": ('AR', "US", "America/Chicago"),
    "revolution concert house|garden city": ('ID', "US", "America/Denver"),
    "garden amp|garden grove": ('CA', "US", "America/Los_Angeles"),
    "gibson area hospital|gibson city": ('IL', "US", "America/Chicago"),
    "bank of new hampshire pavilion|gilford": ('NH', "US", "America/New_York"),
    "heartland events center|grand island": ('NE', "US", "America/Chicago"),
    "center for the arts|grass valley": ('CA', "US", "America/Los_Angeles"),
    "james lumber performing arts center|grayslake": ('IL', "US", "America/Chicago"),
    "groton hill music center|groton": ('MA', "US", "America/New_York"),
    "space ballroom hamden|hamden": ('CT', "US", "America/New_York"),
    "point of the bluff vineyards concert pavilion|hammondsport": ('NY', "US", "America/New_York"),
    "delaware state fair fairgrounds|harrington": ('DE', "US", "America/New_York"),
    "first baptist church|hazard": ('KY', "US", "America/New_York"),
    "pavilion at ravinia|highland park": ('IL', "US", "America/Chicago"),
    "fairbanks lawn at hollywood forever|hollywood": ('CA', "US", "America/Los_Angeles"),  # Hollywood, CA
    "hard rock live at seminole hard rock hotel|hollywood": ('FL', "US", "America/New_York"),  # Hollywood, FL
    "carnegie library music hall of homestead|homestead": ('PA', "US", "America/New_York"),
    "kansas state fair grandstand|hutchinson": ('KS', "US", "America/Chicago"),
    "windjammer|isle of palms": ('SC', "US", "America/New_York"),
    "tuacahn amphitheatre outdoors|ivins": ('UT', "US", "America/Denver"),
    "cactus petes resort casino amphitheater|jackpot": ('NV', "US", "America/Denver"),
    "capital region mu health care amphitheater|jefferson city": ('MO', "US", "America/Chicago"),
    "kahilu theatre|kamuela": ('HI', "US", "Pacific/Honolulu"),
    "colonial theatre|keene": ('NH', "US", "America/New_York"),
    "arcadia live theatre|kerrville": ('TX', "US", "America/Chicago"),
    "beak and skiff apple orchards|la fayette": ('NY', "US", "America/New_York"),
    "la mirada theatre for perf arts|la mirada": ('CA', "US", "America/Los_Angeles"),
    "astro|la vista": ('NE', "US", "America/Chicago"),
    "american music theatre|lancaster": ('PA', "US", "America/New_York"),
    "tally ho theater|leesburg": ('VA', "US", "America/New_York"),
    "koussevitzky music shed at tanglewood|lenox": ('MA', "US", "America/New_York"),
    "seiji ozawa hall at tanglewood|lenox": ('MA', "US", "America/New_York"),
    "pine creek lodge|livingston": ('MT', "US", "America/Denver"),
    "ellen eccles theatre|logan": ('UT', "US", "America/Denver"),
    "blue fcu arena|loveland": ('CO', "US", "America/Denver"),
    "rialto theater|loveland": ('CO', "US", "America/Denver"),
    "northwest washington fair and event center|lynden": ('WA', "US", "America/Los_Angeles"),
    "vixen front lounge|mchenry": ('IL', "US", "America/Chicago"),
    "fillmore|miami beach": ('FL', "US", "America/New_York"),
    "liv miami|miami beach": ('FL', "US", "America/New_York"),
    "north beach bandshell|miami beach": ('FL', "US", "America/New_York"),
    "milton theatre delaware|milton": ('DE', "US", "America/New_York"),
    "tjs corral outdoor events center at carson valley inn casino|minden": ('NV', "US", "America/Los_Angeles"),
    "miramar regional park amphitheatre|miramar": ('FL', "US", "America/New_York"),
    "world s only corn palace|mitchell": ('SD', "US", "America/Chicago"),
    "city winery hudson valley indoors|montgomery": ('NY', "US", "America/New_York"),
    "newberry opera house|newberry": ('SC', "US", "America/New_York"),
    "ferguson center for the arts|newport news": ('VA', "US", "America/New_York"),
    "tioga downs|nichols": ('NY', "US", "America/New_York"),
    "amp at craig ranch regional park|north las vegas": ('NV', "US", "America/Los_Angeles"),
    "alabama theatre at barefoot landing|north myrtle beach": ('SC', "US", "America/New_York"),
    "house of blues|north myrtle beach": ('SC', "US", "America/New_York"),
    "riviera theatre|north tonawanda": ('NY', "US", "America/New_York"),
    "toyota arena|ontario": ('CA', "US", "America/Los_Angeles"),
    "post og orange grove|orange grove": ('TX', "US", "America/Chicago"),
    "oxnard performing arts and conv ctr|oxnard": ('CA', "US", "America/Los_Angeles"),
    "theater at mgm national harbor|oxon hill": ('MD', "US", "America/New_York"),
    "carson center for the performing arts|paducah": ('KY', "US", "America/Chicago"),
    "rose bowl|pasadena": ('CA', "US", "America/Los_Angeles"),
    "patchogue theatre|patchogue": ('NY', "US", "America/New_York"),
    "mystic theatre|petaluma": ('CA', "US", "America/Los_Angeles"),
    "phoenix theater petaluma|petaluma": ('CA', "US", "America/Los_Angeles"),
    "colonial theatre|phoenixville": ('PA', "US", "America/New_York"),
    "capitol theatre|port chester": ('NY', "US", "America/New_York"),
    "jeanne rimsky theater at landmark on main street|port washington": ('NY', "US", "America/New_York"),
    "quarry park amphitheater|rocklin": ('CA', "US", "America/Los_Angeles"),
    "walmart arkansas music pavilion|rogers": ('AR', "US", "America/Chicago"),
    "liberty|roswell": ('NM', "US", "America/Denver"),
    "stiefel theatre for perf arts|salina": ('KS', "US", "America/Chicago"),
    "santa monica pier santa monica|santa monica": ('CA', "US", "America/Los_Angeles"),
    "broadview stage at spac|saratoga springs": ('NY', "US", "America/New_York"),
    "kewadin casino sault ste marie dreammakers theater|sault sainte marie": ('MI', "US", "America/New_York"),
    "backyard seguin|seguin": ('TX', "US", "America/Chicago"),
    "goat ranch|south greenfield": ('MO', "US", "America/Chicago"),
    "nugget event center at nugget casino resort|sparks": ('NV', "US", "America/Los_Angeles"),
    "palace theatre|st paul": ('MN', "US", "America/Chicago"),
    "mill river park stamford|stamford": ('CT', "US", "America/New_York"),
    "stamford center for the arts palace theatre|stamford": ('CT', "US", "America/New_York"),
    "stockbridge amphitheater|stockbridge": ('GA', "US", "America/New_York"),
    "sherman theater|stroudsburg": ('PA', "US", "America/New_York"),
    "kit carson park|taos": ('NM', "US", "America/Denver"),
    "fred kavli theatre at the thousand oaks civic arts plaza|thousand oaks": ('CA', "US", "America/Los_Angeles"),
    "cadence bank arena|tupelo": ('MS', "US", "America/Chicago"),
    "full throttle saloon|vale": ('SD', "US", "America/Denver"),
    "grand theater|wausau": ('WI', "US", "America/Chicago"),
    "big e xfinity arena|west springfield": ('MA', "US", "America/New_York"),
    "levitt pavilion|westport": ('CT', "US", "America/New_York"),
    "young auditorium|whitewater": ('WI', "US", "America/Chicago"),
    "williamsport community arts center|williamsport": ('PA', "US", "America/New_York"),
    "shenandoah downs county fair association|woodstock": ('VA', "US", "America/New_York"),
    "stadium theatre performing arts centre and conservatory|woonsocket": ('RI', "US", "America/New_York"),
    "capitol theatre|yakima": ('WA', "US", "America/Los_Angeles"),

    # --- US: city name shared by several places; checked by hand so a wrong city match is overwritten ---
    "empire live|albany": ('NY', "US", "America/New_York"),
    "arlington improv|arlington": ('TX', "US", "America/Chicago"),
    "miller theater|augusta": ('GA', "US", "America/New_York"),
    "stampede aurora|aurora": ('CO', "US", "America/Denver"),  # Aurora, CO
    "alys robinson stephens pac|birmingham": ('AL', "US", "America/Chicago"),
    "saturn|birmingham": ('AL', "US", "America/Chicago"),
    "bluebird nightclub|bloomington": ('IN', "US", "America/New_York"),  # Bloomington, IN
    "castle theatre|bloomington": ('IL', "US", "America/Chicago"),  # Bloomington, IL
    "paramount center for the arts tn|bristol": ('TN', "US", "America/New_York"),
    "clay center|charleston": ('WV', "US", "America/New_York"),
    "blue note|columbia": ('MO', "US", "America/Chicago"),  # Columbia, MO
    "central bank downtown live|columbia": ('MO', "US", "America/Chicago"),  # Columbia, MO
    "senate|columbia": ('SC', "US", "America/New_York"),  # Columbia, SC
    "woodlawn church|columbia": ('MS', "US", "America/Chicago"),  # Columbia, MS
    "celeste center at ohio expo center|columbus": ('OH', "US", "America/New_York"),
    "columbus funny bone|columbus": ('OH', "US", "America/New_York"),
    "rumba cafe|columbus": ('OH', "US", "America/New_York"),
    "schottenstein center|columbus": ('OH', "US", "America/New_York"),
    "skullys music diner|columbus": ('OH', "US", "America/New_York"),
    "capitol center for the arts|concord": ('NH', "US", "America/New_York"),
    "norton center|danville": ('KY', "US", "America/New_York"),
    "eddie s attic|decatur": ('GA', "US", "America/New_York"),
    "state theatre|easton": ('PA', "US", "America/New_York"),
    "alpine valley music theatre|elkhorn": ('WI', "US", "America/Chicago"),
    "memoire boston|everett": ('MA', "US", "America/New_York"),  # Everett, MA
    "crown coliseum|fayetteville": ('NC', "US", "America/New_York"),
    "magic bag ferndale|ferndale": ('MI', "US", "America/New_York"),
    "pines theater at look park|florence": ('MA', "US", "America/New_York"),
    "heard county performing arts center|franklin": ('GA', "US", "America/New_York"),
    "pacific steel and recycling four seasons arena|great falls": ('MT', "US", "America/Denver"),
    "oaklawn park|hot springs": ('AR', "US", "America/Chicago"),
    "paramount|huntington": ('NY', "US", "America/New_York"),
    "albatross|jacksonville": ('FL', "US", "America/New_York"),
    "five|jacksonville": ('FL', "US", "America/New_York"),
    "florida theatre|jacksonville": ('FL', "US", "America/New_York"),
    "times union center performance arts moran theater|jacksonville": ('FL', "US", "America/New_York"),
    "madrid theatre|kansas city": ('MO', "US", "America/Chicago"),
    "bottleneck|lawrence": ('KS', "US", "America/Chicago"),
    "granada theater ks|lawrence": ('KS', "US", "America/Chicago"),
    "lied center lawrence|lawrence": ('KS', "US", "America/Chicago"),
    "lexington s cary memorial hall|lexington": ('MA', "US", "America/New_York"),
    "lied center lincoln|lincoln": ('NE', "US", "America/Chicago"),
    "long beach amphitheater|long beach": ('CA', "US", "America/Los_Angeles"),
    "eastside bowl|madison": ('TN', "US", "America/Chicago"),  # Madison, TN (Nashville area)
    "orpheum theatre|madison": ('WI', "US", "America/Chicago"),
    "overture center|madison": ('WI', "US", "America/Chicago"),
    "lincoln theatre|marion": ('VA', "US", "America/New_York"),  # Marion, VA
    "marion palace theatre|marion": ('OH', "US", "America/New_York"),  # Marion, OH
    "soaring eagle outdoor arena|mount pleasant": ('MI', "US", "America/New_York"),
    "stress factory comedy club|new brunswick": ('NJ', "US", "America/New_York"),
    "midland theatre|newark": ('OH', "US", "America/New_York"),  # Newark, OH
    "new jersey performing arts center|newark": ('NJ', "US", "America/New_York"),  # Newark, NJ
    "chartway arena at the ted constant convocation center|norfolk": ('VA', "US", "America/New_York"),
    "infinity hall norfolk|norfolk": ('CT', "US", "America/New_York"),  # Norfolk, CT
    "norva|norfolk": ('VA', "US", "America/New_York"),
    "performing arts center at ocean city convention center|ocean city": ('MD', "US", "America/New_York"),
    "lyric|oxford": ('MS', "US", "America/Chicago"),
    "eccles center for the performing arts|park city": ('UT', "US", "America/Denver"),  # Park City, UT
    "hartman arena|park city": ('KS', "US", "America/Chicago"),  # Park City, KS
    "saenger theatre fl|pensacola": ('FL', "US", "America/Chicago"),
    "aura|portland": ('ME', "US", "America/New_York"),  # Portland, ME
    "crystal ballroom|portland": ('OR', "US", "America/Los_Angeles"),  # Portland, OR
    "live at madrid s|portland": ('ME', "US", "America/New_York"),  # Portland, ME
    "merrill auditorium|portland": ('ME', "US", "America/New_York"),  # Portland, ME
    "portland house of music and events maine|portland": ('ME', "US", "America/New_York"),  # Portland, ME
    "realm|portland": ('OR', "US", "America/Los_Angeles"),  # Portland, OR
    "revolution hall|portland": ('OR', "US", "America/Los_Angeles"),  # Portland, OR
    "star theater|portland": ('OR', "US", "America/Los_Angeles"),  # Portland, OR
    "state theatre|portland": ('ME', "US", "America/New_York"),  # Portland, ME
    "atlantic union bank pavilion|portsmouth": ('VA', "US", "America/New_York"),  # Portsmouth, VA
    "seacoast repertory theatre portsmouth|portsmouth": ('NH', "US", "America/New_York"),  # Portsmouth, NH
    "marymoor park|redmond": ('WA', "US", "America/Los_Angeles"),
    "allianz amphitheater at riverfront|richmond": ('VA', "US", "America/New_York"),
    "broadberry|richmond": ('VA', "US", "America/New_York"),
    "canal club|richmond": ('VA', "US", "America/New_York"),
    "national|richmond": ('VA', "US", "America/New_York"),
    "anthology rochester|rochester": ('NY', "US", "America/New_York"),
    "mayo civic center presentation hall|rochester": ('MN', "US", "America/Chicago"),  # Rochester, MN
    "juanita k hammons hall|springfield": ('MO', "US", "America/Chicago"),
    "sangamon auditorium at uis performing arts center|springfield": ('IL', "US", "America/Chicago"),  # Springfield, IL
    "lake tahoe outdoor arena at harveys|stateline": ('NV', "US", "America/Los_Angeles"),
    "troy fairgrounds|troy": ('PA', "US", "America/New_York"),
    "packard music hall|warren": ('OH', "US", "America/New_York"),
    "black cat|washington": ('DC', "US", "America/New_York"),
    "flash|washington": ('DC', "US", "America/New_York"),
    "songbyrd music house|washington": ('DC', "US", "America/New_York"),
    "grand opera house|wilmington": ('DE', "US", "America/New_York"),
}

In [31]:
# 3b-4. Apply the dictionary (it replaces a same-city guess where both exist), then update is_us
loc = pd.DataFrame.from_dict(VENUE_LOCATIONS, orient="index", columns=["state", "country", "timezone"])
use = venues["venue_key"].isin(loc.index) & venues["location_source"].ne("seatgeek")

# same-city guesses that the hand check corrects
was_guess = use & venues["location_source"].eq("seatgeek_city")
changed = was_guess & ((venues["country"] != venues["venue_key"].map(loc["country"]))
                       | (venues["state"].fillna("") != venues["venue_key"].map(loc["state"]).fillna("")))
print(f"Hand check corrected {changed.sum()} same-city guesses:")
print(venues.loc[changed, ["venue_key", "state", "country"]]
      .assign(new_state=lambda d: d["venue_key"].map(loc["state"]),
              new_country=lambda d: d["venue_key"].map(loc["country"])).to_string(index=False))

for c in ["state", "country", "timezone"]:
    venues.loc[use, c] = venues.loc[use, "venue_key"].map(loc[c])
venues.loc[use, "location_source"] = "manual"
venues["is_us"] = venues["country"].map(lambda c: pd.NA if pd.isna(c) else c == "US")

unused = sorted(set(loc.index) - set(venues["venue_key"]))
if unused:
    print(f"\n{len(unused)} dictionary keys not found in venues (check the spelling):", unused)

print("\nWhere each venue's location comes from:")
print(venues.groupby("location_source")["n_shows"].agg(venues="size", shows="sum")
      .assign(pct_shows=lambda d: (100 * d["shows"] / d["shows"].sum()).round(1)).to_string())
left = venues[venues["location_source"].eq("missing")].sort_values("n_shows", ascending=False)
print(f"\nStill missing: {len(left)} venues ({left['n_shows'].sum():,} shows)")
if len(left):
    print(left[["venue_key", "n_shows"]].to_string(index=False))

Hand check corrected 43 same-city guesses:
                                                        venue_key state country new_state new_country
                                                  history|toronto    ON  Canada       NaN      Canada
                                              metropolis|montreal    QC  Canada       NaN      Canada
                                             paramount|huntington    WV      US        NY          US
                                                  pearl|vancouver    BC  Canada       NaN      Canada
                                        orpheum theatre|vancouver    BC  Canada       NaN      Canada
                                      hollywood theatre|vancouver    BC  Canada       NaN      Canada
                                              opera house|toronto    ON  Canada       NaN      Canada
                                             bell centre|montreal    QC  Canada       NaN      Canada
                                       

### 3c. Fix the time zones
`03_build_shows.ipynb` made local time as UTC minus 6 hours for every show. That is off by 1 to 3 hours for most US shows, and much more abroad. It also ignores the clock change on Nov 1. This section:
1. **Checks that `start_utc` really is UTC:** if it is, the busiest start hour (in UTC) at Eastern venues should be about 3 hours earlier than at Pacific venues.
2. **Cleans the venue time zones:** renames old aliases (`US/Eastern`, ...) to standard names, and replaces SeatGeek time zones that don't fit the venue's state or province (for example `America/Regina` for a US venue). A zone fits if it has the same UTC offset in August and in November as one of the state's zones. Venues with a state but no time zone get the state's main zone.
3. **Converts each show to its venue's local time**, then rebuilds `weekday`, `is_weekend`, `month` and `show_hour`. Shows whose venue still has no time zone keep the old UTC-6 estimate. `tz_source` records which one each show used.

In [32]:
# 3c-1. Is start_utc really UTC? Busiest start hour (UTC) at Eastern vs Pacific venues
shows["start_utc"] = pd.to_datetime(shows["start_utc"])
show_tz = shows["venue_key"].map(venues.set_index("venue_key")["timezone"])
east_west = show_tz.where(show_tz.isin(["America/New_York", "America/Los_Angeles"]))
hours = pd.crosstab(shows["start_utc"].dt.hour, east_west)
print(hours.to_string())
print("\nBusiest UTC start hour:", hours.idxmax().to_dict())
# about 3 hours apart (e.g. 23 vs 2) = start_utc is UTC; the same hour = it is already local time

venue_key  America/Los_Angeles  America/New_York
start_utc                                       
0                           94              4093
1                          462               632
2                         2051               286
3                         2603                94
4                          644                69
5                          310                 7
6                           37                 0
7                           31                 0
8                            4                 0
12                           0                 1
13                           1                 4
14                           0                 5
15                           0                21
16                           0                48
17                           5                35
18                          87                72
19                          24                86
20                           5                59
21                  

In [33]:
# 3c-2. Clean the venue time zones
ET, CT, MT, PT = "America/New_York", "America/Chicago", "America/Denver", "America/Los_Angeles"
TZ_ALIASES = {"US/Eastern": ET, "US/Central": CT, "US/Mountain": MT, "US/Pacific": PT,
              "America/Indianapolis": "America/Indiana/Indianapolis", "America/Montreal": "America/Toronto"}
# zones allowed in each state / province; the first one is used as the fix
STATE_TZ = {
    "AL": [CT], "AK": ["America/Anchorage"], "AZ": ["America/Phoenix"], "AR": [CT], "CA": [PT], "CO": [MT],
    "CT": [ET], "DC": [ET], "DE": [ET], "FL": [ET, CT], "GA": [ET], "HI": ["Pacific/Honolulu"], "ID": [MT, PT],
    "IL": [CT], "IN": [ET, CT], "IA": [CT], "KS": [CT, MT], "KY": [ET, CT], "LA": [CT], "ME": [ET], "MD": [ET],
    "MA": [ET], "MI": [ET, CT], "MN": [CT], "MS": [CT], "MO": [CT], "MT": [MT], "NE": [CT, MT], "NV": [PT, MT],
    "NH": [ET], "NJ": [ET], "NM": [MT], "NY": [ET], "NC": [ET], "ND": [CT, MT], "OH": [ET], "OK": [CT],
    "OR": [PT, MT], "PA": [ET], "RI": [ET], "SC": [ET], "SD": [CT, MT], "TN": [CT, ET], "TX": [CT, MT],
    "UT": [MT], "VT": [ET], "VA": [ET], "WA": [PT], "WV": [ET], "WI": [CT], "WY": [MT],
    "ON": ["America/Toronto"], "QC": ["America/Toronto"], "BC": ["America/Vancouver", "America/Dawson_Creek"],
    "AB": ["America/Edmonton"], "SK": ["America/Regina"], "MB": ["America/Winnipeg"],
    "NB": ["America/Moncton"], "NS": ["America/Halifax"], "NL": ["America/St_Johns"],
}
REF = [pd.Timestamp("2026-08-15 12:00", tz="UTC"), pd.Timestamp("2026-11-15 12:00", tz="UTC")]
_offsets = {}
def offsets(zone):
    # UTC offset in August and November (before and after the clock change)
    if zone not in _offsets:
        _offsets[zone] = tuple(t.tz_convert(zone).utcoffset() for t in REF)
    return _offsets[zone]

venues["timezone"] = venues["timezone"].replace(TZ_ALIASES)
has_rule = venues["state"].isin(STATE_TZ.keys())
fits = [offsets(z) in {offsets(a) for a in STATE_TZ[s]} if r and pd.notna(z) else True
        for s, z, r in zip(venues["state"], venues["timezone"], has_rule)]
wrong = has_rule & ~pd.Series(fits, index=venues.index)
no_tz = has_rule & venues["timezone"].isna()

fix = wrong | no_tz
print(f"Time zones replaced: {wrong.sum()} wrong for the state, {no_tz.sum()} missing")
print(venues.loc[fix, ["venue_key", "state", "timezone"]]
      .assign(new_timezone=lambda d: d["state"].map(lambda s: STATE_TZ[s][0])).to_string(index=False))
venues.loc[fix, "timezone"] = venues.loc[fix, "state"].map(lambda s: STATE_TZ[s][0])

Time zones replaced: 9 wrong for the state, 0 missing
                                    venue_key state            timezone        new_timezone
                   rickshaw theatre|vancouver    BC America/Los_Angeles   America/Vancouver
                        park theatre|winnipeg    MB America/Los_Angeles    America/Winnipeg
                    capital ballroom|victoria    BC America/Los_Angeles   America/Vancouver
chan centre for the performing arts|vancouver    BC America/Los_Angeles   America/Vancouver
                       edge pavilion|laughlin    NV     America/Phoenix America/Los_Angeles
    apac arbogast performing arts center|troy    OH     America/Chicago    America/New_York
         fort bend county epicenter|rosenberg    TX      America/Regina     America/Chicago
            red deer memorial centre|red deer    AB      America/Denver    America/Edmonton
         kelowna community theatre bc|kelowna    BC America/Los_Angeles   America/Vancouver


In [34]:
# 3c-3. Convert each show to its venue's local time and rebuild the date features
show_tz = shows["venue_key"].map(venues.set_index("venue_key")["timezone"])
start_utc = shows["start_utc"].dt.tz_localize("UTC")
shows["start_local"] = shows["start_utc"] - pd.Timedelta(hours=6)    # old estimate, kept where no time zone
for zone, idx in shows.groupby(show_tz).groups.items():
    shows.loc[idx, "start_local"] = start_utc.loc[idx].dt.tz_convert(zone).dt.tz_localize(None)
shows["tz_source"] = np.where(show_tz.notna(), "venue", "approx_utc-6")

old = shows[["weekday", "month", "show_hour"]].copy()
local = shows["start_local"]
shows["weekday"] = local.dt.day_name()
shows["is_weekend"] = local.dt.dayofweek.isin([4, 5])                # Friday and Saturday, same as 03
shows["month"] = local.dt.month
shows["show_hour"] = local.dt.hour

print(shows["tz_source"].value_counts().to_string())
print(f"\nChanged by the fix: weekday {(old['weekday'] != shows['weekday']).sum():,} shows | "
      f"month {(old['month'] != shows['month']).sum():,} | show_hour {(old['show_hour'] != shows['show_hour']).sum():,}")
print("\nLocal start hour, all shows:")
print(shows["show_hour"].value_counts().sort_index().to_string())

tz_source
venue           29443
approx_utc-6      170

Changed by the fix: weekday 266 shows | month 14 | show_hour 26,666

Local start hour, all shows:
show_hour
0       247
1         7
2         2
3         3
5         1
6         1
7         1
8         3
9         5
10       19
11      131
12      125
13       75
14      244
15      175
16      146
17      467
18     2595
19    11262
20    11640
21     1447
22      835
23      182


/var/folders/hp/mq_22xwd7013lzv3g8vv2rnm0000gn/T/ipykernel_46696/2380897209.py:12: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  shows["is_weekend"] = local.dt.dayofweek.isin([4, 5])                # Friday and Saturday, same as 03


### 3d. Duplicates in each table (before the join in 07)
1. **Venues:** different `venue_key`s that SeatGeek matched to the same venue (for example "The Fillmore" and "Fillmore"). They are merged into the key with the most shows, and `shows.venue_key` is updated to match.
2. **Artists:** different `artist_key`s that Last.fm matched to the same artist (for example "A.J. Croce" and "AJ Croce"). They are merged the same way. This also matters for the train/test split by artist: otherwise one artist could land in both sets under two keys.
3. **Shows:** the same `event_id` twice, and the same artist at the same venue at the exact same start time under different IDs. Both are true duplicates, and the copy with the most listings is kept. Shows by the same artist at the same venue on the same day but at different times are kept (early and late shows) and only reported.

Both merge lists are printed. If a pair is really two different venues or artists, add it to `KEEP_SEPARATE_VENUES` / `KEEP_SEPARATE_ARTISTS` and re-run.

In [35]:
# 3d-1. Venues: different venue_keys matched to the same SeatGeek venue
KEEP_SEPARATE_VENUES = []   # sg_venue_id values to leave as separate venues, after checking the list
dup_v = venues[venues["sg_venue_id"].notna() & venues.duplicated("sg_venue_id", keep=False)
               & ~venues["sg_venue_id"].isin(KEEP_SEPARATE_VENUES)]
print(f"{dup_v['sg_venue_id'].nunique()} SeatGeek venues have more than one venue_key:")
print(dup_v.sort_values(["sg_venue_id", "n_shows"], ascending=[True, False])
      [["sg_venue_id", "venue_key", "sg_name", "n_shows"]].to_string(index=False))

# keep the key with the most shows and point the others to it
keep_v = dup_v.sort_values("n_shows", ascending=False).drop_duplicates("sg_venue_id").set_index("sg_venue_id")["venue_key"]
venue_map = {k: keep_v[i] for k, i in zip(dup_v["venue_key"], dup_v["sg_venue_id"]) if k != keep_v[i]}
total_v = dup_v.groupby("sg_venue_id")["n_shows"].sum()
kept = venues["venue_key"].isin(keep_v.values)
venues.loc[kept, "n_shows"] = venues.loc[kept, "sg_venue_id"].map(total_v)
venues = venues[~venues["venue_key"].isin(venue_map)].reset_index(drop=True)
shows["venue_key"] = shows["venue_key"].replace(venue_map)
print(f"\nMerged {len(venue_map)} venue_keys; {len(venues):,} venues left")

4 SeatGeek venues have more than one venue_key:
 sg_venue_id                                       venue_key                                       sg_name  n_shows
      1333.0         raising cane s river center|baton rouge                   Raising Cane's River Center        5
      1333.0 raising cane s river center theater|baton rouge                   Raising Cane's River Center        1
      1486.0              harrah s cherokee center|asheville          Harrah's Cherokee Center - Asheville        7
      1486.0     harrahs cherokee center asheville|asheville          Harrah's Cherokee Center - Asheville        1
      5437.0              peoria civic center theatre|peoria                     Peoria Civic Center Arena        9
      5437.0                      peoria civic center|peoria                     Peoria Civic Center Arena        1
    180656.0              dover international speedway|dover The Woodlands of Dover International Speedway        1
    180656.0 woodlands o

In [36]:
# 3d-2. Artists: different artist_keys matched to the same Last.fm artist
KEEP_SEPARATE_ARTISTS = []   # lastfm_name values to leave as separate artists, after checking the list
a = artists[artists["lastfm_status"].eq("ok") & artists["lastfm_name"].notna()
            & ~artists["low_confidence_match"].fillna(False).astype(bool)].copy()
a["lfm"] = a["lastfm_name"].str.lower()
dup_a = a[a.duplicated("lfm", keep=False) & ~a["lastfm_name"].isin(KEEP_SEPARATE_ARTISTS)]
print(f"{dup_a['lfm'].nunique()} Last.fm artists have more than one artist_key:")
with pd.option_context("display.max_rows", None):
    print(dup_a.sort_values(["lfm", "n_shows"], ascending=[True, False])
          [["lastfm_name", "artist_key", "artist_name", "n_shows"]].to_string(index=False))

# keep the key with the most shows and point the others to it
keep_a = dup_a.sort_values("n_shows", ascending=False).drop_duplicates("lfm").set_index("lfm")["artist_key"]
artist_map = {k: keep_a[l] for k, l in zip(dup_a["artist_key"], dup_a["lfm"]) if k != keep_a[l]}
total_a = dup_a.groupby("lfm")["n_shows"].sum()
kept = artists["artist_key"].isin(keep_a.values)
artists.loc[kept, "n_shows"] = artists.loc[kept, "lastfm_name"].str.lower().map(total_a)
# n_venues is left as the kept key's value (the artists table has no venue list to combine)
artists = artists[~artists["artist_key"].isin(artist_map)].reset_index(drop=True)
shows["artist_key"] = shows["artist_key"].replace(artist_map)
print(f"\nMerged {len(artist_map)} artist_keys; {len(artists):,} artists left")

35 Last.fm artists have more than one artist_key:
                        lastfm_name                          artist_key                         artist_name  n_shows
                        8Ball & MJG                       8ball and mjg                       8Ball and MJG        3
                        8Ball & MJG                      8 ball and mjg                      8 Ball and MJG        1
                         A.J. Croce                           a j croce                          A.J. Croce       20
                         A.J. Croce                            aj croce                            AJ Croce        5
                  Alanis Morissette                   alanis morissette                   Alanis Morissette        6
                  Alanis Morissette                          morissette                          Morissette        1
                        Blue Monday                         blue monday                         Blue Monday        2
              

In [37]:
# 3d-3. Shows: repeated event_ids, and the same artist + venue + start time under different IDs
n_start = len(shows)
# the copy with the most listings (then most priced days) comes first, so it is the one kept
shows = shows.sort_values(["listings", "n_priced_days"], ascending=False)

rep_id = shows.duplicated("event_id")
print(f"Repeated event_id: {rep_id.sum()} rows dropped")
shows = shows[~rep_id]

KEY = ["artist_key", "venue_key", "start_utc"]
same_time = shows.duplicated(KEY, keep=False)
print(f"\nSame artist, venue and start time under different event_ids: {same_time.sum()} rows "
      f"in {shows[same_time].groupby(KEY).ngroups} shows")
print(shows[same_time].sort_values(KEY)[["event_id", "name", "venue_key", "start_local", "listings", "price_median"]]
      .head(30).to_string(index=False))
shows = shows.drop_duplicates(KEY)

day = shows["start_local"].dt.date
same_day = shows.assign(day=day).duplicated(["artist_key", "venue_key", "day"], keep=False)
print(f"\nKept, same artist and venue on the same day at different times (early/late shows): {same_day.sum()} shows")

shows = shows.sort_values("start_utc").reset_index(drop=True)
assert shows["event_id"].is_unique
print(f"\nShows: {n_start:,} -> {len(shows):,} ({n_start - len(shows):,} duplicates dropped)")

Repeated event_id: 0 rows dropped

Same artist, venue and start time under different event_ids: 52 rows in 26 shows
                event_id                                    name                                        venue_key         start_local  listings  price_median
6a5eb7f459f0a2bf5fd3be5c                            After - Band                                    vinyl|atlanta 2026-10-17 20:00:00         1            19
6ab6a60cddb7d88234952a13                                   After                                    vinyl|atlanta 2026-10-17 20:00:00         1            19
6a17139795aa981e30835e66                                  Anuhea                              blue note napa|napa 2026-08-09 19:00:00        11           144
6a1c0eef98db3095afbd6930                                  Anuhea                              blue note napa|napa 2026-08-09 19:00:00         5           110
698bb02be2be43ac2df3af0b              Australian Pink Floyd Show outlaw field at the idaho bot

In [38]:
# Save the cleaned tables for 07
from pathlib import Path
DATA_DIR = Path("../data")
shows.to_csv(DATA_DIR / f"shows_clean.csv", index=False)
artists.to_csv(DATA_DIR / f"artists_clean.csv", index=False)
venues.to_csv(DATA_DIR / f"venues_clean.csv", index=False)

# COMMENTING THIS OUT BECAUSE IT GOES INTO 07 WITH PROPER CLEANING/FILTERING
# 4. Non-US shows
KEEP_UNKNOWN_COUNTRY = True   # True = keep shows whose venue SeatGeek couldn't place
NON_US_CITIES = ["London", "Montreal", "Toronto", "Vancouver", "Manchester", "Glasgow",
                 "Dublin", "Mexico City", "Calgary", "Edmonton", "Ottawa", "Winnipeg"]

shows = shows.merge(venues[["venue_key", "is_us", "country"]], on="venue_key", how="left")
is_us = shows["is_us"].map({True: True, False: False, "True": True, "False": False})
unknown = is_us.isna()
foreign_city = shows["city"].isin(NON_US_CITIES)   # backup check for unmatched venues

keep = (is_us == True) | (unknown & ~foreign_city & KEEP_UNKNOWN_COUNTRY)
dropped = shows[~keep]
print(f"Dropping {len(dropped):,} non-US shows; keeping {keep.sum():,}")
print("Most common dropped cities:")
print(dropped["city"].value_counts().head(10).to_string())
shows = shows[keep].drop(columns=["is_us"]).reset_index(drop=True)

# 5. Extreme prices
TARGET = "price_median"   # or "price_min": group decision
PRICE_CAP = 2000          # drop shows priced above this ($)

too_high = shows[TARGET] > PRICE_CAP
print(f"{too_high.sum()} shows with {TARGET} above ${PRICE_CAP:,} ({too_high.mean():.2%} of shows):")
display(shows.loc[too_high, ["name", "venue", "city", "listings", "price_min", "price_median"]]
             .sort_values(TARGET, ascending=False).head(20))
shows = shows[~too_high].reset_index(drop=True)

print(f"\nAfter cleaning: {len(shows):,} shows")
print(shows[TARGET].describe(percentiles=[.05, .25, .5, .75, .95]).round(0).to_string())